# The Reasoning-Effort Trap: recomputing arXiv 2610.34491

Sun et al. (Oct 25, 2026) measured what happens when you turn the reasoning-effort dial up on 8 models across AIME 2026, GPQA Diamond, Terminal-Bench 2.1 and SWE-Bench Pro. This notebook recomputes the paper's headline numbers from the values reported in the paper — no API keys, no model calls, just the arithmetic behind the claims.

In [ ]:
# 1. Overspend on always-solved tasks: cost multipliers vs the lowest effort
multipliers = {
    "Terminal-Bench GPT-5.6-sol (low->xhigh)": 2.51,
    "Terminal-Bench Claude Sonnet 5 (low->xhigh)": 3.04,
    "SWE-Bench Pro GPT-5.6-sol (low->high)": 4.54,
    "SWE-Bench Pro GPT-5.6-luna (low->high)": 5.34,
}
for k, v in multipliers.items():
    print(f"{k}: {v}x")
# SWE-Bench Pro sol in dollars: $0.27 -> $1.24
print("sol dollar ratio:", round(1.24 / 0.27, 2))
assert abs(1.24 / 0.27 - 4.54) < 0.06  # paper rounds to 4.54x
print("Paper max on always-solved tasks: 42x (lowest -> highest effort)")


In [ ]:
# 2. Regressions: higher effort loses tasks lower effort solved
# GPT-5.6-sol, Terminal-Bench, high -> xhigh: 76 -> 73 solved, +5 gained, -8 lost
solved_high, gained, lost = 76, 5, 8
solved_xhigh = solved_high + gained - lost
print("xhigh solved:", solved_xhigh, "of 89")
assert solved_xhigh == 73
# Inkling on GPQA, low -> medium: net +16 hides 5 regressions
net, regressions = 16, 5
print("Inkling GPQA: net +16 =", f"+{net + regressions} gained, -{regressions} lost")
assert net + regressions - regressions == net
# Every one of the 8 models had a nonempty regression set on Terminal-Bench (lowest vs highest)
print("Models with nonempty Terminal-Bench regression set: 8/8")


In [ ]:
# 3. The two cost mechanisms: more compute per call AND longer trajectories
# GPT-5.6-sol, SWE-Bench Pro always-solved: input bill $0.17 -> $0.60, calls 14.7 -> 28.4
print("input-bill ratio:", round(0.60 / 0.17, 2))
print("trajectory-length ratio:", round(28.4 / 14.7, 2))
# The 'None' trap: GPT-5.6-luna, always-solved: 21.5 calls at none vs 5.4 at low, 4.58x cost
print("none-vs-low calls:", round(21.5 / 5.4, 2), "x; cost 4.58x; input bill $0.0021 -> $0.0258")
assert abs(0.0258 / 0.0021 - 12.29) < 0.01
# Shortcut behavior: GPT-5.6-sol reference-fix reads 15.9% (low) -> 75.0% (xhigh)
print("reference-fix read ratio:", round(75.0 / 15.9, 2), "x")
# Verification: Sonnet 5 Terminal-Bench passed-at-both tasks 6.6 -> 12.3 checks
print("verification ratio:", round(12.3 / 6.6, 2), "x — more checking, same outcome")


In [ ]:
# 4. Naive laddering: start low, escalate only after a verified failure
# Terminal-Bench: (fixed-top success, laddering success, cost saving)
ladder = {
    "GPT-5.6-sol": (83.0, 88.6, 17.4),
    "Claude Sonnet 5": (76.1, 83.0, 8.7),
    "Kimi K3": (73.0, 84.3, 30.5),
}
for m, (fixed, lad, save) in ladder.items():
    print(f"{m}: {fixed}% -> {lad}% (+{lad - fixed:.1f}pp), cost -{save}%")
    assert lad > fixed
# Median savings vs fixed highest effort, incl. both attempts on retry:
print("Median laddering savings: AIME 25.3%, GPQA 41.5%, corrected SWE-Bench Pro 15.5%")
print("Self-routing (model picks effort from the description alone): no consistent advantage over random.")


In [ ]:
# 5. Toy laddering simulator: why starting low wins even when retries cost double
import random
random.seed(7)
# Task mix: 60% easy (low solves), 25% medium (needs high), 15% hard (needs max)
# Costs per attempt: low=1, high=4, max=10 units (illustrative, paper-consistent ordering)
def run(policy):
    cost, solved = 0, 0
    for _ in range(10000):
        r = random.random()
        need = "low" if r < 0.60 else ("high" if r < 0.85 else "max")
        price = {"low": 1, "high": 4, "max": 10}
        if policy == "fixed-max":
            cost += 10; solved += 1  # max solves everything in this toy world
        else:  # laddering: low first, escalate only on verified failure
            cost += 1
            if need == "low":
                solved += 1
            elif need == "high":
                cost += 4; solved += 1
            else:
                cost += 10; solved += 1
    return cost, solved
cm, sm = run("fixed-max")
cl, sl = run("laddering")
print(f"fixed-max: cost {cm}, solved {sm}")
print(f"laddering: cost {cl}, solved {sl}, saving {(1 - cl/cm)*100:.1f}%")
assert sl == sm and cl < cm
print("Lesson: you pay max only on the 40% of tasks that need it; the 60% easy majority stays cheap.")
